# Лабораторная работа №2

## Нейросетевой перевод на базе Hugging Face Transformers (MarianMT, NLLB)

**ФИО студента:** _______________________
**Группа:** _______________________
**Вариант:** ______ (тема модуля из ЛР № 1: _______________________)

---

**Цель:** развернуть NMT-пайплайн, исследовать субсловную токенизацию, батчинг, условную
кросс-энтропию и cross-attention, сравнить специализированную EN→RU модель **MarianMT**
с многоязычной **NLLB**.

**Максимум: 20 баллов**

| Критерий | Состав | Баллы |
|---|---|---:|
| Работоспособность пайплайна | MarianMT — 2 · NLLB — 2 · batching/loss/attention — 2 | **6** |
| Корректность лингвистической обработки | языковые коды и токенизация — 2 · объём корпуса (120+ сегментов, 100+ терминов) — 1 · glossary QA — 2 | **5** |
| Анализ ошибок и аргументация | 5+ различий — 2 · интерпретация loss/attention — 2 · вывод — 1 | **5** |
| Оформление и воспроизводимость | результаты и графики — 1 · README — 1 · зависимости — 1 · структура — 1 | **4** |

---

### Порядок выполнения

1. **Runtime → Change runtime type → T4 GPU.** Без GPU перевод 120 сегментов двумя моделями
   займёт часы.
2. Выполните блоки 0–1 (установка, окружение).
3. Блок 2 — подключите **свои** файлы из ЛР № 1 (`segments_en.csv`, `glossary.csv`,
   `aligned_reference.csv`) и убедитесь, что проверка объёма даёт «ГОТОВО».
4. Реализуйте функции в блоках 3–8 вместо маркеров `# TODO`.
5. Запустите блок 9 — все автотесты должны пройти без `AssertionError`.
6. Выполните блоки 10–15: перевод корпуса, QA, сравнение, **графики**, отчёт.
7. Заполните вручную колонки `preferred` и `comment` — автоматический выбор модели их не
   заменяет.

### Обязательный минимум

| Требование | Значение |
|---|---|
| Сегментов в корпусе, переведённых **каждой** моделью | **120+** |
| Терминов в глоссарии, разобранных токенизаторами | **100+** |
| Пар `source` / `reference` для кросс-энтропии | 5+ |
| Пар для анализа cross-attention | 2+ |
| Режимы инференса | последовательный и пакетный |
| Содержательных различий с аргументацией | 5+ |
| Графиков в отчёте | 4+ |

> Если в ЛР № 1 корпус меньше, расширьте исходный учебный модуль и глоссарий до требуемого
> объёма **до** начала работы: все дальнейшие выводы опираются на статистику, а на 15
> сегментах она недостоверна.

### Две особенности актуальных версий библиотек

* **`attn_implementation="eager"`.** В `transformers` 5.x модели по умолчанию используют
  ускоренное внимание (SDPA/Flash), которое не сохраняет веса: `outputs.cross_attentions`
  окажется пустым, и блок 7 упадёт с `IndexError: tuple index out of range`.
* **Предупреждение `max_new_tokens` / `max_length`.** Безобидно (приоритет у
  `max_new_tokens`), но зашумляет вывод — логирование понижается в блоке 1.

## Блок 0. Установка зависимостей

`torch` в Colab предустановлен. Ставим `transformers`, `sentencepiece` (нужен токенизатору
NLLB) и `sacremoses` (предобработка MarianMT).

In [ ]:
!pip install -q "transformers>=4.40" sentencepiece sacremoses

## Блок 1. Окружение и структура каталогов

Ячейка готова — менять её не нужно. Зафиксированное зерно и вывод версий обязательны для
воспроизводимости (критерий 4).

In [ ]:
from __future__ import annotations

import json
import platform
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

transformers.logging.set_verbosity_error()   # убирает предупреждение о max_length

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE = Path("/content/lab02") if Path("/content").exists() else Path.cwd() / "lab02"
DATA = BASE / "data"
RESULTS = BASE / "results"
FIGURES = BASE / "figures"
for folder in (DATA, RESULTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MARIAN_NAME = "Helsinki-NLP/opus-mt-en-ru"
NLLB_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG, TGT_LANG = "eng_Latn", "rus_Cyrl"

MIN_SEGMENTS, MIN_TERMS = 120, 100

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

ENVIRONMENT = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-",
}
for key, value in ENVIRONMENT.items():
    print(f"{key:>13}: {value}")
print(f"{'base_dir':>13}: {BASE}")
if DEVICE.type != "cuda":
    print("\nВНИМАНИЕ: GPU не подключён. Runtime -> Change runtime type -> T4 GPU.")

## Блок 2. Данные своего варианта из ЛР № 1

Нужны три файла:

| Файл | Обязательные колонки | Объём |
|---|---|---|
| `data/segments_en.csv` | `segment_id`, `source` | не менее 120 сегментов |
| `data/glossary.csv` | `en`, `ru` | не менее 100 терминов |
| `data/aligned_reference.csv` | `segment_id`, `source`, `reference` | те же сегменты |

Способ 1 — загрузить файлы вручную (диалог выбора появится после запуска ячейки).
Способ 2 — раскомментировать монтирование Google Drive и указать путь к своей папке.
Если ничего не загружено, создаётся **демонстрационный** мини-набор: он годится только для
проверки автотестов, сдавать работу на нём нельзя.

In [ ]:
# --- Способ 1: загрузка файлов с компьютера ---------------------------------
try:
    from google.colab import files

    print("Выберите segments_en.csv, glossary.csv, aligned_reference.csv "
          "(или закройте диалог, чтобы использовать демо-данные)")
    uploaded = files.upload()
    for name, content in uploaded.items():
        (DATA / name).write_bytes(content)
except Exception as error:  # noqa: BLE001 - вне Colab или диалог закрыт
    print("Загрузка пропущена:", error.__class__.__name__)

# --- Способ 2: Google Drive ------------------------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# for name in ("segments_en.csv", "glossary.csv", "aligned_reference.csv"):
#     shutil.copy(f"/content/drive/MyDrive/lab01/{name}", DATA / name)

# --- Демо-набор на случай отсутствия файлов --------------------------------
DEMO_PARALLEL = [
    ("A variable stores a value.", "Переменная хранит значение."),
    ("A condition controls which branch is executed.",
     "Условие определяет, какая ветвь выполняется."),
    ("A loop repeats a block of instructions.", "Цикл повторяет блок инструкций."),
    ("A function can return a value.", "Функция может возвращать значение."),
]
DEMO_GLOSSARY = [("variable", "переменная"), ("value", "значение"),
                 ("loop", "цикл"), ("function", "функция")]

if not (DATA / "segments_en.csv").exists():
    print("\nФайлы ЛР № 1 не найдены — создаётся демонстрационный набор.")
    pd.DataFrame([{"segment_id": f"s{i:03d}", "source": src}
                  for i, (src, _) in enumerate(DEMO_PARALLEL, 1)]
                 ).to_csv(DATA / "segments_en.csv", index=False, encoding="utf-8")
    pd.DataFrame([{"segment_id": f"s{i:03d}", "source": src, "reference": ref}
                  for i, (src, ref) in enumerate(DEMO_PARALLEL, 1)]
                 ).to_csv(DATA / "aligned_reference.csv", index=False, encoding="utf-8")
    pd.DataFrame(DEMO_GLOSSARY, columns=["en", "ru"]
                 ).to_csv(DATA / "glossary.csv", index=False, encoding="utf-8")

segments_df = pd.read_csv(DATA / "segments_en.csv")
glossary_df = pd.read_csv(DATA / "glossary.csv")
aligned_df = pd.read_csv(DATA / "aligned_reference.csv")

# --- проверка структуры (обязательна) --------------------------------------
assert {"segment_id", "source"} <= set(segments_df.columns), "segments_en.csv: не те колонки"
assert {"en", "ru"} <= set(glossary_df.columns), "glossary.csv: нужны колонки en и ru"
assert {"segment_id", "source", "reference"} <= set(aligned_df.columns), \
    "aligned_reference.csv: нужны колонки segment_id, source, reference"
assert len(segments_df) == len(aligned_df), \
    "segments_en.csv и aligned_reference.csv должны содержать одинаковые сегменты"


def dataset_ready(segments: pd.DataFrame, glossary: pd.DataFrame) -> bool:
    """Проверить готовность корпуса к сдаче и напечатать отчёт."""
    checks = [
        ("сегментов", len(segments), MIN_SEGMENTS),
        ("терминов", len(glossary), MIN_TERMS),
    ]
    ready = True
    for label, actual, required in checks:
        status = "ГОТОВО" if actual >= required else "НЕДОСТАТОЧНО"
        ready &= actual >= required
        print(f"  {label:<10} {actual:>4} / {required:<4} {status}")
    return ready


print(f"\nПроверка объёма корпуса:")
READY = dataset_ready(segments_df, glossary_df)
print("\nСтатус:", "корпус соответствует требованиям"
      if READY else "работа на этих данных к сдаче НЕ принимается")
print(f"Средняя длина сегмента: {segments_df['source'].str.split().str.len().mean():.1f} слов")
segments_df.head()

## Блок 3. Загрузка моделей (шаги 1–2, 8)

**Подсказки.** `AutoTokenizer.from_pretrained` / `AutoModelForSeq2SeqLM.from_pretrained`;
модель переносится на устройство методом `.to(device)` и переводится в режим инференса
`.eval()` — без этого включён dropout и результат нестабилен. Для NLLB токенизатору
передаются `src_lang` и `tgt_lang`.

**Обязательно:** передайте `attn_implementation="eager"` в `from_pretrained`, иначе блок 7
не получит веса внимания. Старые версии `transformers` этого аргумента не знают — оберните
вызов в `try / except (TypeError, ValueError)`.

In [ ]:
def load_seq2seq(model_name: str, device: torch.device,
                 eager_attention: bool = True,
                 **tokenizer_kwargs: Any) -> Tuple[Any, Any]:
    """Загрузить токенизатор и модель seq2seq и подготовить её к инференсу.

    Args:
        model_name: идентификатор модели на Hugging Face Hub.
        device: устройство, на которое переносится модель.
        eager_attention: загружать модель с ``attn_implementation="eager"``; необходимо,
            чтобы ``output_attentions=True`` возвращал веса внимания.
        **tokenizer_kwargs: дополнительные параметры токенизатора (``src_lang``, ``tgt_lang``).

    Returns:
        Кортеж ``(tokenizer, model)``; модель должна быть на ``device`` и в режиме ``eval``.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 4. Перевод сегментов (шаги 4–5, 9)

**Подсказки.** Токенизация пакета: `tokenizer(list_of_texts, return_tensors="pt",
padding=True, truncation=True).to(device)`. Генерация — под `torch.no_grad()`, декодирование —
`tokenizer.batch_decode(..., skip_special_tokens=True)`.

Для NLLB направление перевода задаётся аргументом `forced_bos_token_id`, который получают как
`tokenizer.convert_tokens_to_ids("rus_Cyrl")`. Без него язык результата не гарантирован.

In [ ]:
def translate_batch(texts: Sequence[str], tokenizer: Any, model: Any,
                    device: torch.device, num_beams: int = 4,
                    max_new_tokens: int = 128,
                    forced_bos_token_id: Optional[int] = None) -> List[str]:
    """Перевести список сегментов одним пакетом.

    Args:
        texts: сегменты исходного языка.
        tokenizer: токенизатор модели.
        model: модель seq2seq в режиме ``eval``.
        device: устройство инференса.
        num_beams: ширина луча при декодировании.
        max_new_tokens: ограничение длины порождаемой последовательности.
        forced_bos_token_id: принудительный первый токен целевого языка (для NLLB);
            если ``None``, аргумент в ``generate`` не передаётся.

    Returns:
        Список переводов, порядок соответствует порядку ``texts``.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def timed_sequential(texts: Sequence[str], tokenizer: Any, model: Any,
                     device: torch.device, **kwargs: Any) -> Tuple[List[str], float]:
    """Перевести сегменты по одному и вернуть ``(переводы, время в секундах)``.

    Используйте ``time.perf_counter()``; прогревочный вызов делается вне замера.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def timed_batched(texts: Sequence[str], tokenizer: Any, model: Any,
                  device: torch.device, batch_size: int = 8,
                  **kwargs: Any) -> Tuple[List[str], float]:
    """Перевести сегменты пачками по ``batch_size`` и вернуть ``(переводы, время)``."""
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 5. Субсловная токенизация (шаги 3, 10)

**Подсказка.** `tokenizer(term, add_special_tokens=False)["input_ids"]`, затем
`tokenizer.convert_ids_to_tokens(ids)`. Специальные токены отключаются намеренно: иначе в
разборе появятся `</s>` и код языка.

In [ ]:
def tokenize_terms(terms: Sequence[str], tokenizer: Any, label: str) -> pd.DataFrame:
    """Разобрать термины токенизатором модели.

    Args:
        terms: список терминов на исходном языке.
        tokenizer: токенизатор Hugging Face.
        label: имя модели, попадает в колонку ``model``.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``model`` (str), ``term`` (str), ``n_words`` (int), ``n_tokens`` (int),
        ``tokens`` (str — токены через пробел),
        ``is_word_level`` (bool — число токенов совпало с числом слов термина).
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 6. Условная кросс-энтропия (шаг 6)

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T}\log P\left(y_t \mid y_{1:t-1}, x\right)$$

**Подсказка.** Эталон передаётся в токенизатор аргументом `text_target=...`; при прямом вызове
модели `model(**batch)` поле `outputs.loss` уже содержит усреднённую кросс-энтропию.

In [ ]:
def sequence_cross_entropy(source: str, target: str, tokenizer: Any,
                           model: Any, device: torch.device) -> float:
    """Вычислить условную кросс-энтропию целевой строки при данном источнике.

    Args:
        source: сегмент на исходном языке.
        target: перевод, вероятность которого оценивается.
        tokenizer: токенизатор модели.
        model: модель seq2seq.
        device: устройство вычислений.

    Returns:
        Средний ``-log P`` на токен (положительное число типа ``float``).
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 7. Cross-attention (шаг 7)

**Подсказки.** Вызов модели с `output_attentions=True, return_dict=True`;
`outputs.cross_attentions` — кортеж по слоям, каждый элемент имеет форму
`[batch, heads, target_len, source_len]`. Головы усредняются: `attn[0].mean(dim=0)`.
В `batch["labels"]` могут быть значения `-100` (игнорируемые позиции) — их надо отфильтровать
перед `convert_ids_to_tokens`.

**Обязательно:** если `outputs.cross_attentions` пуст, поднимите `RuntimeError` с понятным
текстом. Пустой кортеж означает, что модель загружена без `attn_implementation="eager"`.

In [ ]:
def extract_cross_attention(source: str, target: str, tokenizer: Any, model: Any,
                            device: torch.device, layer: int = -1
                            ) -> Tuple[pd.DataFrame, Tuple[int, ...]]:
    """Извлечь усреднённую по головам матрицу cross-attention.

    Args:
        source: сегмент исходного языка.
        target: целевая строка.
        tokenizer: токенизатор модели.
        model: модель seq2seq.
        device: устройство вычислений.
        layer: индекс слоя декодера (по умолчанию последний).

    Returns:
        Кортеж ``(frame, shape)``:
        ``frame`` — ``pd.DataFrame``, индекс — целевые токены, колонки — исходные токены,
        значения — веса внимания; ``shape`` — исходная форма тензора
        ``(batch, heads, target_len, source_len)``.

    Raises:
        RuntimeError: если модель не вернула веса внимания.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 8. Терминологический QA и сводная таблица (шаги 12–13)

**Подсказки.** Проверка идёт по глоссарию ЛР № 1. Из-за русской морфологии точное вхождение
даёт ложные срабатывания («сложность» → «сложности»), поэтому сравнивайте нормализованные
формы: достаточно грубого отсечения окончаний. Каждое найденное нарушение затем
подтверждается вручную.

In [ ]:
def term_compliance(source: str, translation: str,
                    glossary: pd.DataFrame) -> List[Dict[str, str]]:
    """Найти термины глоссария, которые есть в источнике, но отсутствуют в переводе.

    Args:
        source: сегмент на исходном языке.
        translation: перевод сегмента.
        glossary: таблица глоссария с колонками ``en`` и ``ru``.

    Returns:
        Список нарушений; каждое — словарь с ключами ``source_term`` и ``preferred_ru``.
        Пустой список означает, что все встреченные термины переданы.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def write_jsonl(records: Iterable[Dict[str, Any]], path: Path) -> Path:
    """Записать записи в JSON Lines: один объект на строку, UTF-8, ``ensure_ascii=False``."""
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def build_comparison_table(marian_records: Sequence[Dict[str, Any]],
                           nllb_records: Sequence[Dict[str, Any]],
                           references: Sequence[str]) -> pd.DataFrame:
    """Собрать сводную таблицу сравнения двух моделей.

    Args:
        marian_records: записи перевода MarianMT (ключи ``id``, ``source``, ``translation``).
        nllb_records: записи перевода NLLB в том же порядке.
        references: эталонные переводы в том же порядке.

    Returns:
        ``pd.DataFrame`` с колонками ``segment_id``, ``source``, ``marian``, ``nllb``,
        ``reference``, ``identical`` (bool), ``preferred`` (пустая строка),
        ``comment`` (пустая строка). Две последние заполняются вручную.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 9. Автотесты

Запускайте по порядку **после** реализации функций. Тест 1 работает без моделей; тесты 2–6
скачивают веса (первый запуск — несколько минут).

In [ ]:
# --- Тест 1: функции, не требующие моделей -------------------------------
demo_glossary = pd.DataFrame([("loop", "цикл"), ("function", "функция")],
                             columns=["en", "ru"])

assert term_compliance("A loop repeats instructions.", "Цикл повторяет инструкции.",
                       demo_glossary) == [], "Термин передан — нарушений быть не должно"

violations = term_compliance("A loop repeats instructions.",
                             "Итерация повторяет инструкции.", demo_glossary)
assert len(violations) == 1, f"Ожидается 1 нарушение, получено {len(violations)}"
assert violations[0]["source_term"] == "loop", "Не то поле source_term"
assert violations[0]["preferred_ru"] == "цикл", "Не то поле preferred_ru"

assert term_compliance("A function returns a value.", "Функции возвращают значение.",
                       demo_glossary) == [], \
    "Словоизменение («функции») не должно считаться нарушением"

assert term_compliance("Nothing relevant here.", "Ничего существенного.",
                       demo_glossary) == [], "Термина нет в источнике — проверять нечего"

path = write_jsonl([{"id": "s001", "source": "A", "translation": "Б", "model": "m"}],
                   RESULTS / "_selftest.jsonl")
lines = Path(path).read_text(encoding="utf-8").strip().split("\n")
assert len(lines) == 1, "JSONL: одна запись — одна строка"
assert json.loads(lines[0])["translation"] == "Б", "JSONL: кириллица должна сохраняться как есть"
Path(path).unlink()

table = build_comparison_table(
    [{"id": "s001", "source": "A loop.", "translation": "Цикл."}],
    [{"id": "s001", "source": "A loop.", "translation": "Цикл."}],
    ["Цикл."])
for column in ["segment_id", "source", "marian", "nllb", "reference",
               "identical", "preferred", "comment"]:
    assert column in table.columns, f"В сводной таблице нет колонки '{column}'"
assert bool(table.loc[0, "identical"]), "Одинаковые переводы -> identical = True"
assert table.loc[0, "preferred"] == "", "Колонка preferred заполняется вручную, а не кодом"
print("Тест 1 (функции без моделей) — OK")

In [ ]:
# --- Тест 2: MarianMT ----------------------------------------------------
marian_tokenizer, marian_model = load_seq2seq(MARIAN_NAME, DEVICE)

assert marian_model.training is False, "Модель должна быть переведена в режим eval()"
assert str(next(marian_model.parameters()).device).startswith(DEVICE.type), \
    "Модель не перенесена на выбранное устройство"
assert getattr(marian_model.config, "_attn_implementation", "eager") == "eager", \
    "Модель загружена не в режиме eager — блок 7 не получит веса внимания"

outputs = translate_batch(["A loop repeats a block of instructions.",
                           "A function can return a value."],
                          marian_tokenizer, marian_model, DEVICE)
assert len(outputs) == 2, "Число переводов должно совпадать с числом входных сегментов"
assert all(text.strip() for text in outputs), "Пустой перевод"
assert any("а" <= char <= "я" for char in outputs[0].lower()), \
    "Ожидается перевод на русский язык"
print("Тест 2 (MarianMT) — OK")
for text in outputs:
    print(" ", text)

In [ ]:
# --- Тест 3: токенизация -------------------------------------------------
tokens_df = tokenize_terms(glossary_df["en"].tolist(), marian_tokenizer, "MarianMT")

assert isinstance(tokens_df, pd.DataFrame), "tokenize_terms должна вернуть DataFrame"
assert len(tokens_df) == len(glossary_df), "Разобраны не все термины"
for column in ["model", "term", "n_words", "n_tokens", "tokens", "is_word_level"]:
    assert column in tokens_df.columns, f"Нет колонки '{column}'"
assert (tokens_df["n_tokens"] >= 1).all(), "Число токенов не может быть нулевым"
assert tokens_df["tokens"].str.strip().ne("").all(), "Пустая строка токенов"
assert not tokens_df["tokens"].str.contains("</s>").any(), \
    "Спецтокены должны быть отключены (add_special_tokens=False)"
print("Тест 3 (токенизация) — OK")
tokens_df.head()

In [ ]:
# --- Тест 4: кросс-энтропия ----------------------------------------------
source_demo = aligned_df.loc[0, "source"]
reference_demo = aligned_df.loc[0, "reference"]

loss_value = sequence_cross_entropy(source_demo, reference_demo,
                                    marian_tokenizer, marian_model, DEVICE)
assert isinstance(loss_value, float), "Функция должна возвращать float, а не тензор"
assert loss_value > 0, "Кросс-энтропия положительна"
assert np.isfinite(loss_value), "Получено inf/nan — проверьте передачу text_target"

loss_wrong = sequence_cross_entropy(source_demo, "Совершенно посторонний текст про погоду.",
                                    marian_tokenizer, marian_model, DEVICE)
assert loss_wrong > loss_value, \
    "Нерелевантная строка должна получать больший loss, чем эталон"
print(f"Тест 4 (кросс-энтропия) — OK: эталон {loss_value:.4f}, шум {loss_wrong:.4f}")

In [ ]:
# --- Тест 5: cross-attention ---------------------------------------------
frame, shape = extract_cross_attention(source_demo, reference_demo,
                                       marian_tokenizer, marian_model, DEVICE)

assert isinstance(frame, pd.DataFrame), "Первым элементом возвращается DataFrame"
assert len(shape) == 4, "Форма тензора: (batch, heads, target_len, source_len)"
assert shape[0] == 1, "Ожидается batch = 1"
assert frame.shape[0] == len(frame.index), "Строки — целевые токены"
assert frame.shape[1] == len(frame.columns), "Колонки — исходные токены"
assert frame.shape[1] <= shape[3] and frame.shape[0] <= shape[2], \
    "Размер матрицы не должен превышать форму тензора"
assert float(frame.values.min()) >= 0.0, "Веса внимания неотрицательны"
print(f"Тест 5 (cross-attention) — OK: shape={shape}, матрица {frame.shape}")
frame.iloc[:5, :6].round(3)

In [ ]:
# --- Тест 6: NLLB и языковые коды ----------------------------------------
nllb_tokenizer, nllb_model = load_seq2seq(NLLB_NAME, DEVICE,
                                          src_lang=SRC_LANG, tgt_lang=TGT_LANG)
RUS_ID = nllb_tokenizer.convert_tokens_to_ids(TGT_LANG)

assert isinstance(RUS_ID, int) and RUS_ID > 0, \
    f"Код языка '{TGT_LANG}' не найден — проверьте написание"

nllb_output = translate_batch(["A loop repeats a block of instructions."],
                              nllb_tokenizer, nllb_model, DEVICE,
                              forced_bos_token_id=RUS_ID)[0]
assert nllb_output.strip(), "Пустой перевод NLLB"
assert any("а" <= char <= "я" for char in nllb_output.lower()), \
    "forced_bos_token_id не задан — язык результата не гарантирован"
print("Тест 6 (NLLB) — OK")
print(" ", nllb_output)

## Блок 10. Батчинг и время инференса (шаг 5)

Замерьте оба режима на всех своих сегментах и сохраните `results/timing.csv`. Первый вызов не
включайте в замер: он содержит инициализацию вычислителя. Полезно сравнить несколько размеров
пачки (4, 8, 16) — выигрыш насыщается.

In [ ]:
sources = segments_df["source"].tolist()

_ = translate_batch(sources[:2], marian_tokenizer, marian_model, DEVICE)  # прогрев

# TODO: получите seq_out/seq_time через timed_sequential
# TODO: получите переводы и время для batch_size = 4, 8, 16 через timed_batched
# TODO: соберите timing_df с колонками
#       model, mode, batch_size, n_segments, total_s, per_segment_s, device, speedup
# TODO: сохраните в RESULTS / "timing.csv" (encoding="utf-8-sig")
# TODO: напечатайте ускорение и число совпавших переводов между режимами

In [ ]:
# ГРАФИК 1. Время инференса
# TODO: столбчатая диаграмма total_s по режимам (последовательный, bs=4, bs=8, bs=16)
# TODO: подпишите оси, добавьте заголовок и значения над столбцами
# TODO: сохраните figure.savefig(FIGURES / "timing.png") и вызовите plt.show()

**Что получено (заполните после запуска):** время последовательного и пакетного режима,
ускорение, число сегментов, где режимы дали разный перевод. Объясните, почему выигрыш
насыщается при росте `batch_size`.

## Блок 11. Полный перевод модуля двумя моделями (шаг 11)

In [ ]:
# TODO: переведите ВСЕ сегменты моделью MarianMT -> список записей вида
#       {"id": ..., "source": ..., "translation": ..., "model": MARIAN_NAME}
# TODO: переведите те же сегменты моделью NLLB (не забудьте forced_bos_token_id=RUS_ID)
# TODO: сохраните write_jsonl(...) в RESULTS / "results_marian.jsonl" и "results_nllb.jsonl"
# TODO: выведите первые 4 тройки EN / Marian / NLLB для визуальной сверки
# Подсказка: используйте батчинг, иначе 120 сегментов на двух моделях займут много времени

**Что получено (заполните после запуска):** число переведённых сегментов каждой моделью
и 2–3 примера расхождений, замеченных при беглом просмотре.

## Блок 12. Токенизация обеими моделями и её визуализация (шаг 10)

In [ ]:
# TODO: разберите все термины глоссария токенизатором NLLB (tokenize_terms)
# TODO: объедините с таблицей MarianMT и сохраните RESULTS / "tokenization_10_terms.csv"
# TODO: напечатайте классы токенизаторов, размеры словарей и среднее число токенов
# TODO: выведите 3 термина, где разбиения моделей различаются сильнее всего

In [ ]:
# ГРАФИК 2. Сравнение дробления терминов
# TODO: сгруппированная столбчатая диаграмма n_tokens (Marian vs NLLB)
#       по 20-30 самым показательным терминам — все 100+ на одном графике нечитаемы
# TODO: сохраните FIGURES / "tokenization_comparison.png"

**Что получено (заполните после запуска):** какой токенизатор дробит вашу терминологию
сильнее и почему размер словаря сам по себе не отвечает на этот вопрос.

## Блок 13. Кросс-энтропия, внимание и их визуализация (шаги 6–7)

In [ ]:
# TODO: посчитайте loss эталона и loss собственного вывода модели минимум для 5 пар
# TODO: соберите loss_df (segment_id, source, reference, marian_output,
#       loss_reference, loss_own_output, gap) и сохраните RESULTS / "loss_5_pairs.csv"

# ГРАФИК 3. Сравнение loss эталона и вывода модели
# TODO: сгруппированная столбчатая диаграмма по сегментам, сохранить FIGURES / "loss.png"

In [ ]:
# TODO: извлеките cross-attention минимум для 2 пар (extract_cross_attention)
# TODO: сохраните матрицы в RESULTS / f"attention_marian_{segment_id}.csv"
# TODO: соберите attention_summary (segment_id, tensor_shape, n_heads, target_len,
#       source_len, mean_max_weight) -> RESULTS / "attention_summary.csv"

# ГРАФИК 4. Тепловая карта cross-attention
# TODO: plt.imshow(frame.values), подписи осей — токены, colorbar,
#       сохранить FIGURES / "cross_attention.png"

**Что получено (заполните после запуска):** соотношение loss эталона и собственного
вывода, сегмент с наибольшим разрывом и его причина; форма тензора внимания и то, что видно
на тепловой карте (диагональ, отклонения, вес служебных токенов).

## Блок 14. Терминологический QA и сводная таблица (шаги 12–13)

In [ ]:
# TODO: для каждой пары (сегмент, модель) вызовите term_compliance
# TODO: соберите glossary_qa с колонками
#       segment_id, model, source_term, preferred_ru, translation, confirmed, comment
#       (confirmed/comment заполняются ВРУЧНУЮ: yes / no — false positive)
# TODO: сохраните RESULTS / "glossary_qa.csv"
# TODO: посчитайте долю соблюдения терминологии; знаменатель — число ВХОЖДЕНИЙ терминов
#       в исходный текст, а не число всех пар «сегмент × термин»

# ГРАФИК 5. Термины, которые чаще всего нарушаются
# TODO: горизонтальная столбчатая диаграмма по терминам с разбивкой по моделям,
#       сохранить FIGURES / "glossary_qa.png"

In [ ]:
# TODO: постройте comparison = build_comparison_table(...)
# TODO: отберите кандидатов на разбор: сегменты, где модели разошлись или есть нарушения
# TODO: заполните preferred и comment минимум для 5 сегментов — ВРУЧНУЮ, с обоснованием
# TODO: сохраните RESULTS / "comparison.csv"

**Что получено (заполните после запуска):** число кандидатов в нарушения по каждой
модели, доля соблюдения глоссария, самые проблемные термины и сколько из нарушений оказались
ложными срабатываниями после ручной проверки.

## Блок 15. Анализ различий и ошибок (5+ случаев)

Заполните таблицу вручную по фактическому выводу моделей. Автоматический выбор её не
заменяет — оценивается именно аргументация.

| № | Сегмент | Наблюдение | Тип различия | Предпочтительный вариант и почему |
|--:|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |

**Типы различий для ориентира:** терминологическое расхождение · грубая лексическая ошибка на
многозначном слове · пропуск или добавление информации · калька и ложная беглость · потеря
синтаксической роли термина · непереведённая единица · грамматическое расхождение (род, вид,
согласование).

## Блок 16. Отчёт, зависимости и выгрузка

In [ ]:
# TODO: сохраните requirements.txt с версиями torch, transformers, pandas, numpy
# TODO: соберите report.md по 10 пунктам из README (среда, модели, токенизация, переводы,
#       loss, attention, время, различия, glossary compliance, вывод) со ссылками на графики
# TODO: заархивируйте BASE и скачайте архив:
#       archive = shutil.make_archive(str(BASE / "lab02_results"), "zip", root_dir=BASE)
#       from google.colab import files; files.download(archive)

In [ ]:
# --- проверка готовности к сдаче -----------------------------------------
required_results = ["results_marian.jsonl", "results_nllb.jsonl", "comparison.csv",
                    "tokenization_10_terms.csv", "loss_5_pairs.csv",
                    "attention_summary.csv", "timing.csv", "glossary_qa.csv"]
required_figures = ["timing.png", "tokenization_comparison.png", "loss.png",
                    "cross_attention.png", "glossary_qa.png"]

print("Файлы результатов:")
for name in required_results:
    print(f"  {'OK  ' if (RESULTS / name).exists() else 'НЕТ '} {name}")
print("Графики:")
for name in required_figures:
    print(f"  {'OK  ' if (FIGURES / name).exists() else 'НЕТ '} {name}")
print("Отчёт и зависимости:")
for name in ("report.md", "requirements.txt"):
    print(f"  {'OK  ' if (BASE / name).exists() else 'НЕТ '} {name}")
print(f"\nОбъём корпуса: сегментов {len(segments_df)} / {MIN_SEGMENTS}, "
      f"терминов {len(glossary_df)} / {MIN_TERMS}")

## Блок 17. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Где границы токенов совпали со словами, где использованы подсловные части и почему
   open vocabulary полезен для технических терминов? Почему подсловный токен нельзя считать
   морфемой?
2. Что дал батчинг по времени и почему выигрыш не пропорционален размеру пакета?
3. Как соотносятся loss эталона и loss собственного вывода модели? Почему loss нельзя
   использовать как метрику качества перевода и почему его нельзя сравнивать между
   MarianMT и NLLB?
4. Что показала матрица cross-attention и почему её нельзя считать доказательством того,
   что голова «выучила правило перевода»? Что произойдёт, если загрузить модель без
   `attn_implementation="eager"`?
5. Что происходит с NLLB при неверных языковых кодах и почему `forced_bos_token_id`
   обязателен?
6. Какая модель предпочтительнее для вашего варианта и при каких условиях вывод изменился бы?
7. Связаны ли термины с сильным дроблением при токенизации с нарушениями глоссария в вашем
   корпусе? Подтвердите цифрами.

**Ваши выводы:**

_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_02_Фамилия.ipynb` (доступ «для всех по ссылке»), все ячейки
   выполнены, вывод сохранён.
2. Каталог `results/`: `results_marian.jsonl`, `results_nllb.jsonl`, `comparison.csv`,
   `tokenization_10_terms.csv`, `loss_5_pairs.csv`, `attention_summary.csv`, `timing.csv`,
   `glossary_qa.csv`.
3. Каталог `figures/` — не менее четырёх графиков.
4. `report.md` и `requirements.txt`.